# Mini-Among Us 2D — Dataset Loader & Per-Game Preview

Clicking **Runtime -> Run all** automatically downloads and previews both exported tables from our GitHub repository (`forknaan/AIML2-Project`):
1. **`keystroke_trajectories.csv`** — 10 Hz continuous 2D movement, WASD keystrokes, line-of-sight visibility, cooldowns, and `lights_out` sabotage state.
2. **`meeting_features.csv`** — Per-player meeting features combining continuous round movement behavior, structured Quick Chat claims/vouches/accusations, reaction times (`rt_ms`), Logical Agent flags, and votes.

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

REPO_RAW = "https://raw.githubusercontent.com/forknaan/AIML2-Project/main/data"
TRAJ_URL = f"{REPO_RAW}/keystroke_trajectories.csv"
MEET_URL = f"{REPO_RAW}/meeting_features.csv"

traj_df = pd.read_csv(TRAJ_URL)
meet_df = pd.read_csv(MEET_URL)

print(f"Loaded keystroke_trajectories.csv: {traj_df.shape[0]:,} rows x {traj_df.shape[1]} columns across {traj_df['game_id'].nunique()} games")
print(f"Loaded meeting_features.csv      : {meet_df.shape[0]:,} rows x {meet_df.shape[1]} columns across {meet_df['game_id'].nunique()} games with meetings")

In [ ]:
# Per-Game Summary Table across all recorded games
summary_rows = []
for gid, g_traj in traj_df.groupby("game_id", sort=True):
    g_meet = meet_df[meet_df["game_id"] == gid]
    human_pids = sorted(g_traj.loc[g_traj["is_human"] == 1, "player_id"].unique())
    impostors = sorted(g_traj.loc[g_traj["role"] == "IMPOSTOR", "player_id"].unique())
    summary_rows.append({
        "game_id": gid,
        "duration_sec": round(float(g_traj["t"].max()), 1),
        "traj_rows_10hz": len(g_traj),
        "meetings": int(g_meet["meeting_id"].nunique()) if len(g_meet) else 0,
        "meeting_rows": len(g_meet),
        "max_tasks_done": int(g_traj["tasks_completed_total"].max()),
        "lights_out_ticks": int(g_traj.groupby("t")["lights_out"].max().sum()),
        "impostor": ", ".join(impostors),
        "human_players": ", ".join(human_pids) if human_pids else "None",
    })

per_game_df = pd.DataFrame(summary_rows)
print(f"=== Per-Game Summary ({len(per_game_df)} Games) ===")
display(per_game_df)

print("\n=== Role & Human/Bot Tick Breakdown ===")
display(traj_df.groupby(["role", "is_human"])["player_id"].count().rename("tick_rows").to_frame())

In [ ]:
print("=== Sample Rows: keystroke_trajectories.csv ===")
display(traj_df.head(10))

print("\n=== Sample Rows: meeting_features.csv ===")
display(meet_df.head(10))

In [ ]:
# Per-Game 2D Trajectory Subplots (all games displayed on the 1200x800 6-room map)
ROOMS = {
    "Cafeteria": (450, 55, 750, 325),
    "Medbay": (80, 70, 340, 310),
    "Admin": (860, 70, 1120, 310),
    "Reactor": (80, 490, 340, 730),
    "Storage": (460, 480, 740, 740),
    "Electrical": (860, 490, 1120, 730),
}
PLAYER_COLORS = {
    "P0": "#38bdf8",
    "P1": "#fb923c",
    "P2": "#4ade80",
    "P3": "#f472b6",
    "P4": "#facc15",
}

game_ids = sorted(traj_df["game_id"].unique())
n_games = len(game_ids)
ncols = min(3, max(1, n_games))
nrows = math.ceil(n_games / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 4.6 * nrows), squeeze=False)
for idx, gid in enumerate(game_ids):
    ax = axes[idx // ncols][idx % ncols]
    ax.set_facecolor("#0b1120")
    for rname, (x1, y1, x2, y2) in ROOMS.items():
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor="#475569", facecolor="#152238", alpha=0.85)
        ax.add_patch(rect)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, rname, color="#64748b", fontsize=7.5, ha="center", va="center", weight="bold")

    g_df = traj_df[traj_df["game_id"] == gid]
    for pid, grp in g_df.groupby("player_id"):
        role_short = "IMP" if grp["role"].iloc[0] == "IMPOSTOR" else "CREW"
        kind = "H" if grp["is_human"].iloc[0] == 1 else "B"
        ax.plot(
            grp["x"],
            grp["y"],
            color=PLAYER_COLORS.get(pid, "#ffffff"),
            label=f"{pid} ({role_short},{kind})",
            linewidth=1.5,
            alpha=0.88,
        )

    dur = g_df["t"].max()
    n_meet = int(meet_df.loc[meet_df["game_id"] == gid, "meeting_id"].nunique())
    ax.set_xlim(0, 1200)
    ax.set_ylim(800, 0)
    ax.set_title(f"Game {idx + 1}: {gid}\n{dur:.1f}s | {len(g_df):,} rows | {n_meet} meeting(s)", fontsize=9.5, weight="bold")
    ax.legend(loc="lower left", fontsize=6.8, framealpha=0.75, ncol=2)
    ax.set_xticks([])
    ax.set_yticks([])

# Hide any unused subplot axes
for idx in range(n_games, nrows * ncols):
    axes[idx // ncols][idx % ncols].axis("off")

plt.suptitle("Mini-Among Us 2D — All Recorded Game Trajectories (10 Hz)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()